# 09 · Two-Stage Model Experiment

**Amaç:** Cascade fikrini disiplinli test etmek. Stage-1 OOF skorlarıyla candidate pool oluşturulur (time-aware fold'larla), Stage-2 bu havuzda eğitilir.

**Tasarım:** Time-aware out-of-fold scoring → selection bias'ı azaltır.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

from src.data.loader import load_validated

import joblib, json
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from src.data.loader import load_validated
from src.features.instant import add_derived
from src.features.historical import add_label_free_aggregates
from src.models.split import time_based_split
from src.models.metrics import summary

In [ ]:
df, _ = load_validated()
df = add_derived(df)
df = add_label_free_aggregates(df)
split = time_based_split(df)
print('Train:', len(split.train), 'Test:', len(split.test))

## Stage-1 OOF scoring on train (time-aware folds)

In [ ]:
from src.models.train import _feature_columns, _model_hgb
num, cat = _feature_columns(df, drop_demographic=True)
feat = num + cat

# 4 zaman-bazlı fold (expanding window)
import numpy as np
train = split.train.sort_values('TransactionDate').reset_index(drop=True)
n = len(train)
fold_edges = [int(n*0.25), int(n*0.5), int(n*0.75), n]
start = 0
oof = np.zeros(n)
for i, end in enumerate(fold_edges):
    if start == 0:
        start = end  # ilk fold için train yok, atla
        continue
    pipe = _model_hgb(num, cat)
    pipe.fit(train.iloc[:start][feat], train.iloc[:start]['IsFraudTransaction'])
    oof[start:end] = pipe.predict_proba(train.iloc[start:end][feat])[:,1]
    print(f'fold {i}: trained on {start}, scored {start}:{end}')
    start = end

## Stage-2 — top %5 OOF havuzda eğit

In [ ]:
k = max(1, int(0.05 * n))
order = np.argsort(-oof)
cand_idx = order[:k]
cand = train.iloc[cand_idx]
print('Candidate pool size:', len(cand), '| fraud içinde:', int(cand['IsFraudTransaction'].sum()))
pipe2 = _model_hgb(num, cat)
pipe2.fit(cand[feat], cand['IsFraudTransaction'])

## Test üzerinde cascade çalıştır

In [ ]:
# Stage 1 — tüm test'i skorla (full-train üzerinde eğitilen modeli kullanmak için tek seferlik fit)
pipe1_full = _model_hgb(num, cat)
pipe1_full.fit(train[feat], train['IsFraudTransaction'])
s1 = pipe1_full.predict_proba(split.test[feat])[:,1]
# Stage 2 — yalnız top-%5'e uygulanır; geri kalanda stage 1 skoru kullanılır.
thr_idx = np.argsort(-s1)[:int(0.05*len(split.test))]
mask = np.zeros(len(split.test), dtype=bool); mask[thr_idx] = True
proba = s1.copy()
proba[mask] = pipe2.predict_proba(split.test[feat].iloc[thr_idx])[:,1]

import json
cascade = summary(split.test['IsFraudTransaction'].to_numpy(), proba,
                  total_days_for_alerts=(split.test['TransactionDate'].max()-split.test['TransactionDate'].min()).days)
single = summary(split.test['IsFraudTransaction'].to_numpy(), s1,
                 total_days_for_alerts=(split.test['TransactionDate'].max()-split.test['TransactionDate'].min()).days)
print('Single model PR-AUC:', single['core']['pr_auc'])
print('Cascade PR-AUC:    ', cascade['core']['pr_auc'])

## Sonuç
- Time-aware OOF Stage-1 + Stage-2 cascade test edildi.
- Tek-modeli geçemediği durumda cascade benimsenmez. Sonuç ve karar `reports/model_comparison.md` içinde.